<a href="https://colab.research.google.com/github/marilynhruza-creator/Estadistica-/blob/main/clase_practica_3_descriptivas__agrupados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Preparación de los Datos Agrupados**

Primero, construimos la tabla de frecuencias con las marcas de clase ($X_i$) y las frecuencias acumuladas ($F_i$).

In [ ]:
# Usamos el dataset y calculamos intervalos con Sturges
calidad_aire <- airquality
k <- nclass.Sturges(calidad_aire$Wind)
intervalos <- cut(calidad_aire$Wind, breaks = k)

In [ ]:
# Creamos la tabla de frecuencias básica
frec_abs <- as.vector(table(intervalos))

# Obtenemos los límites inferiores (L_inf) y superiores (L_sup) de cada intervalo
limites <- levels(intervalos)
# Extraemos los números usando expresiones regulares
lim_inf <- as.numeric(sub("^\\((.+),(.+)\\]$", "\\1", limites))
lim_sup <- as.numeric(sub("^\\((.+),(.+)\\]$", "\\2", limites))

In [ ]:
# 1. Calcular la Marca de Clase (Xi) -> El punto medio del intervalo
marca_clase <- (lim_inf + lim_sup) / 2

In [ ]:
# 2. Frecuencia acumulada (Fi)
frec_acum <- cumsum(frec_abs)

In [ ]:
# Consolidamos nuestra tabla de datos agrupados
tabla_agrupada <- data.frame(
  Intervalo = limites,
  Lim_Inf = lim_inf,
  Lim_Sup = lim_sup,
  Xi = marca_clase,
  fi = frec_abs,
  Fi = frec_acum
)

print(tabla_agrupada)

## Notación importante para realizar los calculos

- $X_i$: marca de clase.
- $f_i$: frecuencia absoluta.
- $F_i$: frecuencia acumulada.
- $L_i$: límite inferior de la clase correspondiente.
- $A$: amplitud del intervalo.
- $N$: número total de observaciones.
- $\bar{x}$: media aritmética.
- $Me$: mediana.
- $Mo$: moda.
- $S^2$: varianza.
- $S$: desvío estándar.

# **Cálculo de la Media para Datos Agrupados**

La fórmula matemática de la media para datos agrupados es:$$\bar{X} = \frac{\sum (X_i \cdot f_i)}{N}$$En R lo calculamos multiplicando vectorialmente las marcas de clase por sus frecuencias:

In [ ]:
# Total de datos (N)
N <- sum(tabla_agrupada$fi)

# Cálculo de la Media
media_agrupados <- sum(tabla_agrupada$Xi * tabla_agrupada$fi) / N

cat("Media para datos agrupados:", media_agrupados, "\n")
cat("Media real de los datos sin agrupar:", mean(calidad_aire$Wind), "(para comparar)\n")

la funcion cat () permite imprimir los resultados obtenidos considerando una oracion que nosotros definimos

# **Cálculo de la Mediana para Datos Agrupados**

La fórmula requiere identificar el intervalo mediano (el primer intervalo donde $F_i \ge N/2$) y aplicar:$$Me = L_i + \left( \frac{\frac{N}{2} - F_{i-1}}{f_i} \right) \cdot A$$Donde $L_i$ es el límite inferior, $F_{i-1}$ es la frecuencia acumulada anterior, $f_i$ es la frecuencia del intervalo y $A$ es la amplitud del intervalo.

In [ ]:
# 1. Posición de la mediana
posicion <- N / 2

# 2. Encontrar la fila del intervalo mediano
fila_mediana <- which(tabla_agrupada$Fi >= posicion)[1]

# 3. Extraer los componentes para la fórmula
L_i <- tabla_agrupada$Lim_Inf[fila_mediana]
F_anterior <- if(fila_mediana == 1) 0 else tabla_agrupada$Fi[fila_mediana - 1]
f_i <- tabla_agrupada$fi[fila_mediana]
Amplitud <- tabla_agrupada$Lim_Sup[fila_mediana] - tabla_agrupada$Lim_Inf[fila_mediana]

# 4. Aplicar la fórmula
mediana_agrupados <- L_i + ((posicion - F_anterior) / f_i) * Amplitud

cat("Mediana para datos agrupados:", mediana_agrupados, "\n")
cat("Mediana real de los datos sin agrupar:", median(calidad_aire$Wind), "\n")

# **Cálculo de la Moda para Datos Agrupados**

Para la moda, buscamos el intervalo modal (el que tiene la mayor frecuencia absoluta $f_i$) y aplicamos:$$Mo = L_i + \left( \frac{f_i - f_{i-1}}{(f_i - f_{i-1}) + (f_i - f_{i+1})} \right) \cdot A$$

In [ ]:
# 1. Encontrar la fila del intervalo modal (frecuencia más alta)
fila_modal <- which.max(tabla_agrupada$fi)

# 2. Extraer componentes
L_i_mo <- tabla_agrupada$Lim_Inf[fila_modal]
f_mo <- tabla_agrupada$fi[fila_modal]
f_anterior_mo <- if(fila_modal == 1) 0 else tabla_agrupada$fi[fila_modal - 1]
f_siguiente_mo <- if(fila_modal == nrow(tabla_agrupada)) 0 else tabla_agrupada$fi[fila_modal + 1]
Amplitud_mo <- tabla_agrupada$Lim_Sup[fila_modal] - tabla_agrupada$Lim_Inf[fila_modal]

# 3. Aplicar la fórmula
d1 <- f_mo - f_anterior_mo
d2 <- f_mo - f_siguiente_mo

moda_agrupados <- L_i_mo + (d1 / (d1 + d2)) * Amplitud_mo

#moda_s/agrupar
install.packages("modeest")
library("modeest")
moda_s_agrupar<- modeest::mfv(calidad_aire$Wind) #moda
cat("Moda para datos agrupados:", moda_agrupados, "\n")
cat("Moda real de los datos sin agrupar:", moda_s_agrupar, "\n")

# **Medidas de posición relativas**

## Cuartiles

Los cuartiles para datos agrupados se calculan como:

$$
Q_k=L_i+\left(\frac{\frac{kN}{4}-F_{i-1}}{f_i}\right)\times A
$$

con

$$
k=1,2,3
$$

In [ ]:
#################################
# CUARTILES
#################################

calcular_cuartil <- function(k){

  posicion <- k*N/4

  fila <- which(tabla_agrupada$Fi >= posicion)[1]

  Li <- tabla_agrupada$Lim_Inf[fila]

  Fi_ant <- if(fila==1) 0 else tabla_agrupada$Fi[fila-1]

  fi <- tabla_agrupada$fi[fila]

  A <- tabla_agrupada$Lim_Sup[fila]-
       tabla_agrupada$Lim_Inf[fila]

  Q <- Li + ((posicion-Fi_ant)/fi)*A

  return(Q)

}

Q1 <- calcular_cuartil(1)
Q2 <- calcular_cuartil(2)
Q3 <- calcular_cuartil(3)

cat("Q1 =",Q1,"\n")
cat("Q2 =",Q2,"\n")
cat("Q3 =",Q3,"\n")

## Deciles

Los deciles se calculan mediante:

$$
D_k=L_i+\left(\frac{\frac{kN}{10}-F_{i-1}}{f_i}\right)\times A
$$

con

$$
k=1,\ldots,9
$$

In [ ]:
#################################
# DECILES
#################################

calcular_decil <- function(k){

  posicion <- k*N/10

  fila <- which(tabla_agrupada$Fi>=posicion)[1]

  Li <- tabla_agrupada$Lim_Inf[fila]

  Fi_ant <- if(fila==1)0 else tabla_agrupada$Fi[fila-1]

  fi <- tabla_agrupada$fi[fila]

  A <- tabla_agrupada$Lim_Sup[fila]-
       tabla_agrupada$Lim_Inf[fila]

  D <- Li + ((posicion-Fi_ant)/fi)*A

  return(D)

}

D1 <- calcular_decil(1)
D5 <- calcular_decil(5)
D9 <- calcular_decil(9)

cat("D1 =",D1,"\n")
cat("D5 =",D5,"\n")
cat("D9 =",D9,"\n")

## Percentiles

Los percentiles se obtienen mediante:

$$
P_k=L_i+\left(\frac{\frac{kN}{100}-F_{i-1}}{f_i}\right)\times A
$$

con

$$
k=1,\ldots,99
$$

In [ ]:
#################################
# PERCENTILES
#################################

calcular_percentil <- function(k){

  posicion <- k*N/100

  fila <- which(tabla_agrupada$Fi>=posicion)[1]

  Li <- tabla_agrupada$Lim_Inf[fila]

  Fi_ant <- if(fila==1)0 else tabla_agrupada$Fi[fila-1]

  fi <- tabla_agrupada$fi[fila]

  A <- tabla_agrupada$Lim_Sup[fila]-
       tabla_agrupada$Lim_Inf[fila]

  P <- Li + ((posicion-Fi_ant)/fi)*A

  return(P)

}

P25 <- calcular_percentil(25)
P50 <- calcular_percentil(50)
P75 <- calcular_percentil(75)

cat("P25 =",P25,"\n")
cat("P50 =",P50,"\n")
cat("P75 =",P75,"\n")

# **Medidas de dispersion**

## Varianza para datos agrupados

La varianza muestral para datos agrupados es:

$$
S^2=\frac{\sum_{i=1}^{k}f_i(X_i-\bar{x})^2}{N-1}
$$

In [ ]:
#################################
# VARIANZA
#################################

Varianza <- sum(tabla_agrupada$fi*
                  (tabla_agrupada$Xi-media_agrupados)^2)/(N-1)

cat("Varianza =",Varianza,"\n")

## Desvío estándar

$$
S=\sqrt{S^2}
$$

In [ ]:
#################################
# DESVIO ESTANDAR
#################################

Desvio <- sqrt(Varianza)

cat("Desvío estándar =",Desvio,"\n")

## Coeficiente de variación

$$
CV=\frac{S}{\bar{x}}\times100
$$

In [ ]:
#################################
# COEFICIENTE DE VARIACION
#################################

CV <- (Desvio/media_agrupados)*100

cat("Coeficiente de variación =",CV,"%\n")

# **Medidas de forma**

## Asimetría de Pearson

$$
As=\frac{3(\bar{x}-Me)}{S}
$$

In [ ]:
#################################
# ASIMETRIA
#################################

Asimetria <- 3*(media_agrupados-mediana_agrupados)/Desvio

cat("Coeficiente de asimetría =",Asimetria,"\n")

##Interpretación
if(Asimetria>0){

  cat("Distribución asimétrica positiva.\n")

}else if(Asimetria<0){

  cat("Distribución asimétrica negativa.\n")

}else{

  cat("Distribución simétrica.\n")

}

## Curtosis

El coeficiente de curtosis se calcula como:

$$
K=\frac{\sum_{i=1}^{k}f_i(X_i-\bar{x})^4}{N\,S^4}
$$

In [ ]:
#################################
# CURTOSIS
#################################

m4 <- sum(tabla_agrupada$fi*
            (tabla_agrupada$Xi-media_agrupados)^4)/N

Curtosis <- m4/(Desvio^4)

cat("Curtosis =",Curtosis,"\n")

##Interpretación
if(Curtosis<3){

  cat("Distribución platicúrtica.\n")

}else if(Curtosis>3){

  cat("Distribución leptocúrtica.\n")

}else{

  cat("Distribución mesocúrtica.\n")

}
